<a href="https://colab.research.google.com/drive/1D-ir6m01Q19pgKi2-QdZGvR2Fa-cKyIE"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

# Can A Model Solve French *Mots Fléchés*? Starter And Baseline

This notebook explores the **Mots Fléchés Définitions | French Crossword Clues** dataset, with 10,000 real French arrow-word clues and hand-checked answers from [La Petite Case](https://www.lapetitecase.fr). It runs as is in Google Colab, the data is downloaded from [Hugging Face](https://huggingface.co/datasets/lapetitecase/mots-fleches-definitions). It then builds a simple baseline for finding the answer from a clue and its number of letters.

Want to try the clues first? Use the free [French Crossword Solver](https://huggingface.co/spaces/lapetitecase/french-crossword-solver).

*Version française.* Ce carnet explore 10 000 définitions de mots fléchés et leurs solutions vérifiées, puis construit une première méthode pour trouver la réponse à partir de la définition et du nombre de lettres. Il fonctionne tel quel dans Google Colab.

**Contents.** Loading the data, a quick tour, length and category statistics, a nearest-neighbour baseline, and how to evaluate your own model (or an LLM).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

URL = 'https://huggingface.co/datasets/lapetitecase/mots-fleches-definitions/resolve/main/french_crossword_clues.csv'
# keep_default_na=False keeps answers such as NA or NONE, which are real words
df = pd.read_csv(URL, keep_default_na=False)
print(df.shape)
df.head(10)

## A Quick Tour

One row per clue → answer pair. A clue can have several valid answers, often of different lengths.

In [ ]:
print('clues:', df.clue_id.nunique(), '| pairs:', len(df))
print(df.split.value_counts().to_string())
df.sample(15, random_state=1)[['clue', 'answer_display', 'length', 'category']]

In [ ]:
multi = df.groupby('clue').answer_display.apply(list)
multi[multi.str.len() >= 3].head(10)

## Answer Lengths And Clue Types

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
df.length.value_counts().sort_index().plot.bar(ax=ax[0], color='#3b3fc4')
ax[0].set_title('Answer length (letters)')
df.drop_duplicates('clue_id').category.value_counts().plot.barh(ax=ax[1], color='#c9cbf3')
ax[1].set_title('Clue category (rule-based)')
plt.tight_layout()

In [ ]:
# the shortest clues are often the hardest, one word with several meanings
d = df.drop_duplicates('clue_id').copy()
d['clue_words'] = d.clue.str.split().str.len()
d.clue_words.value_counts().sort_index().head(8)

## Baseline With Nearest Neighbours On Character N-Grams

For each test clue, find the most similar train clues (TF-IDF on character 2–4 grams, accents removed). Then answer with the first neighbour's answer **that has the requested length**.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

train, test = df[df.split == 'train'], df[df.split == 'test']
queries = (test.groupby(['clue_id', 'length'])
               .agg(clue=('clue', 'first'), answers=('answer', set))
               .reset_index())

vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4), strip_accents='unicode')
X = vec.fit_transform(train.clue)
S = linear_kernel(vec.transform(queries.clue), X)
train_answers = train.answer.to_numpy()

def predict(i, length, k=5):
    out = []
    for j in np.argsort(-S[i])[:500]:
        a = train_answers[j]
        if len(a) == length and a not in out:
            out.append(a)
        if len(out) == k:
            break
    return out

preds = [predict(i, r.length) for i, r in queries.iterrows()]
top1 = np.mean([bool(p) and p[0] in r.answers for p, (_, r) in zip(preds, queries.iterrows())])
top5 = np.mean([any(a in r.answers for a in p) for p, (_, r) in zip(preds, queries.iterrows())])
print(f'queries: {len(queries)}  exact match top-1: {top1:.1%}  top-5: {top5:.1%}')

In [ ]:
queries.assign(prediction=[p[0] if p else '' for p in preds]).sample(12, random_state=2)[['clue', 'length', 'answers', 'prediction']]

Surface similarity solves only a small share of the clues. French grid clues rely on synonyms, wordplay, culture and abbreviations. **This is where language models get interesting.**

## Evaluate Your Own Model Or An LLM

Use this function with any `solver(clue, length)` that returns a list of answers. Answers are compared in grid form, uppercase, without accents, spaces or hyphens.

In [ ]:
import unicodedata, re

def grid_form(s):
    s = s.replace('œ', 'oe').replace('Œ', 'OE').replace('æ', 'ae').replace('Æ', 'AE')
    s = ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')
    return re.sub(r'[^A-Z]', '', s.upper())

def evaluate(solver, queries, k=5):
    hits1 = hitsk = 0
    for _, r in queries.iterrows():
        p = [grid_form(a) for a in solver(r.clue, r.length)][:k]
        hits1 += bool(p) and p[0] in r.answers
        hitsk += any(a in r.answers for a in p)
    return hits1 / len(queries), hitsk / len(queries)

# example prompt for an LLM solver
PROMPT = ('Tu es un expert des mots fléchés. Voici la définition « {clue} ». '
          'Donne 5 réponses possibles de {length} lettres, une par ligne, en majuscules.')
print(PROMPT.format(clue='Lac glaciaire', length=7))

## Going Further

- Every clue page on [lapetitecase.fr](https://www.lapetitecase.fr) lists all answers by number of letters (see the `url` column).
- Try any clue in the free [French Crossword Solver](https://huggingface.co/spaces/lapetitecase/french-crossword-solver) on Hugging Face.
- The companion dataset [Dictionnaire Mots Fléchés](https://huggingface.co/datasets/lapetitecase/dictionnaire-mots-fleches) lists the 2,000 short words that fill French grids.
- The same data is on [Kaggle](https://www.kaggle.com/datasets/cuivrax/mots-fleches-definitions-french-crossword-clues).
- Please credit **La Petite Case** (https://www.lapetitecase.fr) when you use this dataset (CC BY 4.0).